### This is a follow-up to Studio 4, using SVMs to classify 4-top ATLAS events. 
In this notebook, you'll use nested cross-validation to optimize hyperparameters for SVM models, comparing to the benchmark linear model that you trained in Studio 4. You'll also add newly-engineered features to the model and test the resulting performance. 

It accompanies Chapter 4 of the book.

Data for this exercise were kindly provided by [Sascha Caron](https://www.nikhef.nl/~scaron/).

Copyright: Viviana Acquaviva (2023)
Modifications by Julieta Gruszko (2025)

License: [BSD-3-clause](https://opensource.org/license/bsd-3-clause/)




### Group Names:

Henry Holland, Peter Chtcheprov, Ben Tonnesen

In [1]:
import numpy as np
import itertools
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import rc
from sklearn.svm import SVC, LinearSVC # New algorithm!
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_predict, cross_validate, cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold, GridSearchCV
from sklearn import metrics

In [2]:
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_colwidth', 100)
rc('text', usetex=False)

## We'll begin by opening the feature and label files you prepared last week.

Read in features and labels.

In [3]:
features = pd.read_csv('../Data/ParticleID_features.csv', index_col='ID')

In [4]:
features.head()

,MET,METphi,P0_type,P0_E,P0_pt,P0_eta,P0_phi,P1_type,P1_E,P1_pt,P1_eta,P1_phi,P2_type,P2_E,P2_pt,P2_eta,P2_phi,P3_type,P3_E,P3_pt,P3_eta,P3_phi,P4_type,P4_E,P4_pt,P4_eta,P4_phi,P5_type,P5_E,P5_pt,P5_eta,P5_phi,P6_type,P6_E,P6_pt,P6_eta,P6_phi,P7_type,P7_E,P7_pt,P7_eta,P7_phi,P8_type,P8_E,P8_pt,P8_eta,P8_phi,P9_type,P9_E,P9_pt,P9_eta,P9_phi,P10_type,P10_E,P10_pt,P10_eta,P10_phi,P11_type,P11_E,P11_pt,P11_eta,P11_phi,P12_type,P12_E,P12_pt,P12_eta,P12_phi,P13_type,P13_E,P13_pt,P13_eta,P13_phi,P14_type,P14_E,P14_pt,P14_eta,P14_phi,P15_type,P15_E,P15_pt,P15_eta,P15_phi,P16_type,P16_E,P16_pt,P16_eta,P16_phi,P17_type,P17_E,P17_pt,P17_eta,P17_phi
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,62803.5,-1.810010,j,137571.0,128444.0,-0.345744,-0.307112,j,174209.0,127932.0,0.826569,2.332000,b,86788.9,84554.9,-0.180795,2.187970,j,140289.0,76955.8,-1.19933,-1.302800,m+,85230.6,70102.4,-0.645689,-1.659540,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,57594.2,-0.509253,j,161529.0,80458.3,-1.318010,1.402050,j,291490.0,68462.9,-2.126740,-2.582310,e-,44270.1,35139.6,-0.706120,-0.371392,e+,72883.9,26902.2,-1.65386,-3.129630,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,82313.3,1.686840,b,167130.0,113078.0,0.937258,-2.068680,j,102423.0,54922.3,1.226850,0.646589,j,60768.9,36244.3,1.102890,-1.434480,j,77714.0,27801.5,1.68461,1.389690,j,26840.0,24469.3,-0.388937,-1.647260,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30610.8,2.617120,j,112267.0,61383.9,-1.211050,-1.457800,b,40647.8,39472.0,-0.024646,-2.222800,j,201589.0,32978.6,-2.496040,1.137810,j,90096.7,26964.5,1.87132,0.817631,j,28235.4,25887.9,-0.411528,2.024290,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,45153.1,-2.241350,j,178174.0,100164.0,1.166880,-0.018721,j,92351.3,69762.1,0.774114,2.568740,j,61625.2,50086.7,0.652572,-3.012800,j,104193.0,31151.0,1.87641,0.865381,j,746585.0,26219.3,4.041820,-0.874169,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [5]:
features.shape

(5000, 92)

In [6]:
y = np.genfromtxt('../Data/ParticleID_labels.txt', dtype = str)

In [7]:
y

array(['ttbar', 'ttbar', 'ttbar', ..., 'ttbar', '4top', 'ttbar'],
      shape=(5000,), dtype='<U5')

#### As we did last week, we'll turn categorical (string-type) labels into an array, e.g. 0/1.

In [8]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder() #turns categorical into 1 ... N
y = le.fit_transform(y)
target = np.abs(y - 1) #flip the labels, so 4-top is 1 and t/t-bar is 0


In [9]:
y

array([1, 1, 1, ..., 1, 0, 1], shape=(5000,))

In [10]:
features.describe() #Note that this automatically excludes non-numerical type columns

,MET,METphi,P0_E,P0_pt,P0_eta,P0_phi,P1_E,P1_pt,P1_eta,P1_phi,P2_E,P2_pt,P2_eta,P2_phi,P3_E,P3_pt,P3_eta,P3_phi,P4_E,P4_pt,P4_eta,P4_phi,P5_E,P5_pt,P5_eta,P5_phi,P6_E,P6_pt,P6_eta,P6_phi,P7_E,P7_pt,P7_eta,P7_phi,P8_E,P8_pt,P8_eta,P8_phi,P9_E,P9_pt,P9_eta,P9_phi,P10_E,P10_pt,P10_eta,P10_phi,P11_E,P11_pt,P11_eta,P11_phi,P12_E,P12_pt,P12_eta,P12_phi,P13_E,P13_pt,P13_eta,P13_phi,P14_E,P14_pt,P14_eta,P14_phi,P15_type,P15_E,P15_pt,P15_eta,P15_phi,P16_type,P16_E,P16_pt,P16_eta,P16_phi,P17_type,P17_E,P17_pt,P17_eta,P17_phi
count,5000.000000,5000.000000,5.000000e+03,5.000000e+03,5000.000000,5000.000000,4.997000e+03,4.997000e+03,4997.000000,4997.000000,4.950000e+03,4950.000000,4950.000000,4950.000000,4.717000e+03,4717.000000,4717.000000,4717.000000,4.002000e+03,4002.000000,4002.000000,4002.000000,2.871000e+03,2871.00000,2871.000000,2871.000000,1.889000e+03,1889.000000,1889.000000,1889.000000,1.186000e+03,1186.000000,1186.000000,1186.000000,7.290000e+02,729.000000,729.000000,729.000000,4.420000e+02,442.000000,442.000000,442.000000,2.610000e+02,261.000000,261.000000,261.000000,1.270000e+02,127.000000,127.000000,127.000000,5.600000e+01,56.000000,56.000000,56.000000,14.000000,14.000000,14.000000,14.000000,3.000000,3.000000,3.000000,3.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
mean,64071.074332,-0.028916,3.301357e+05,1.540486e+05,-0.039812,-0.003049,2.527799e+05,1.080302e+05,-0.029936,0.007327,2.117980e+05,74863.343131,-0.025104,0.011845,1.805997e+05,57289.049481,0.010723,0.045266,1.780366e+05,48798.018516,0.015167,-0.031312,1.705620e+05,44042.67015,-0.022948,0.014522,1.628825e+05,41151.069666,0.002228,0.006738,1.581409e+05,40250.387015,0.072349,-0.035907,1.596814e+05,40139.289849,0.061654,-0.045868,1.574039e+05,39703.038235,0.118543,0.024249,1.561160e+05,38173.716092,0.029455,0.026422,1.631051e+05,34876.849606,0.206978,-0.001085,1.456600e+05,36151.183929,-0.000879,0.219260,180401.885714,27076.621429,-0.276634,0.595697,205102.000000,24610.900000,-1.333743,1.518666,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
std,60525.122480,1.819257,3.068202e+05,1.149469e+05,1.361762,1.814855,2.638580e+05,8.136261e+04,1.439105,1.828832,2.510361e+05,46309.512365,1.577316,1.802715,2.383403e+05,32013.857623,1.634072,1.812078,2.577958e+05,26252.978520,1.744489,1.784248,2.381745e+05,23510.65367,1.806611,1.811101,2.269341e+05,20988.953157,1.815312,1.771888,2.118782e+05,26556.025657,1.836492,1.796932,2.308620e+05,30074.756789,1.842798,1.788596,2.165489e+05,30502.312276,1.872084,1.826435,2.319016e+05,29324.658352,1.884750,1.753017,2.248603e+05,20433.767238,1.998859,1.949004,1.943657e+05,25861.883410,1.941707,1.910400,262972.929941,3332.496623,2.181392,1.760349,230084.978534,1916.495834,2.661912,1.672151,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
min,290.756000,-3.141010,3.857940e+04,2.825400e+04,-4.110220,-3.140710,1.087540e+04,1.080000e+04,-4.668790,-3.140530,1.221050e+04,10639.800000,-4.520250,-3.141480,1.169190e+04,10818.000000,-4.616550,-3.136130,1.110310e+04,10287.000000,-4.778980,-3.139040,1.070330e+04,10066.90000,-4.930230,-3.140380,1.197700e+04,11260.200000,-4.758150,-3.135630,1.380860e+04,10973.300000,-4.606330,-3.132610,1.119760e+04,10067.900000,-4.814380,-3.136380,1.615530e+04,10183.700000,-4.803880,-3.135910,2.004750e+04,14800.200000,-4.400470,-3.130690,1.780380e+04,12987.900000,-4.447660,-3.139820,2.512510e+04,14836.000000,-4.448760,-2.990730,25937.400000,23170.800000,-4.072000,-2.709890,58941.000000,23025.500000,-3.709270,-0.201051,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25%,24352.375000,-1.619905,1.369522e+05,8.883690e+04,-1.035570,-1.574213,1.007510e+05,6.321840e+04,-1.060500,-1.602460,7.636905e+04,46549.475000,-1.125620,-1.547418,5.999090e+04,36097.700000,-1.121240,-1.518030,5.278370e+04,30891.650000,-1.198468,-1.550615,5.007050e+04,28453.95000,-1.250050,-1.586675,4.695560e+04,27963.500000,-1.231420,-1.475380,4.535515e+04,27140.550000,-1.243962,-1.626688,4.387110e+04,26825.

### Imputing Missing Data
As in Studio 4, we'll keep just the first 4 products and fill any reamining missing values with 0's. You'll test other imputation strategies on the homework. 


In [11]:
features_lim = features[['MET', 'METphi', 'P0_E', 'P0_pt', 'P0_eta', 'P0_phi', 'P1_E', 'P1_pt', 'P1_eta', 'P1_phi', 'P2_E', 'P2_pt', 'P2_eta', 'P2_phi', 'P3_E', 'P3_pt', 'P3_eta', 'P3_phi']]
features_lim = features_lim.fillna(0) #Fill with 0 everywhere there is a NaN
features_lim.describe()

,MET,METphi,P0_E,P0_pt,P0_eta,P0_phi,P1_E,P1_pt,P1_eta,P1_phi,P2_E,P2_pt,P2_eta,P2_phi,P3_E,P3_pt,P3_eta,P3_phi
count,5000.000000,5000.000000,5.000000e+03,5.000000e+03,5000.000000,5000.000000,5.000000e+03,5.000000e+03,5000.000000,5000.000000,5.000000e+03,5000.000000,5000.000000,5000.000000,5.000000e+03,5000.000000,5000.000000,5000.000000
mean,64071.074332,-0.028916,3.301357e+05,1.540486e+05,-0.039812,-0.003049,2.526283e+05,1.079653e+05,-0.029918,0.007323,2.096800e+05,74114.709700,-0.024853,0.011727,1.703778e+05,54046.489280,0.010116,0.042704
std,60525.122480,1.819257,3.068202e+05,1.149469e+05,1.361762,1.814855,2.638514e+05,8.138121e+04,1.438673,1.828283,2.506651e+05,46675.655162,1.569410,1.793678,2.352279e+05,33795.723384,1.587146,1.760070
min,290.756000,-3.141010,3.857940e+04,2.825400e+04,-4.110220,-3.140710,0.000000e+00,0.000000e+00,-4.668790,-3.140530,0.000000e+00,0.000000,-4.520250,-3.141480,0.000000e+00,0.000000,-4.616550,-3.136130
25%,24352.375000,-1.619905,1.369522e+05,8.883690e+04,-1.035570,-1.574213,1.007050e+05,6.320943e+04,-1.059270,-1.599617,7.488228e+04,46165.375000,-1.108390,-1.532478,5.480870e+04,33959.400000,-1.050477,-1.424080
50%,46814.400000,-0.055612,2.263525e+05,1.182015e+05,-0.038731,-0.009037,1.658985e+05,8.581595e+04,-0.056810,0.012737,1.277135e+05,62167.100000,-0.023321,0.006687,9.259335e+04,47278.800000,0.000000,0.000000
75%,83032.350000,1.537323,4.077158e+05,1.771265e+05,0.943598,1.542370,2.999058e+05,1.238520e+05,1.028055,1.601880,2.406498e+05,89065.300000,1.048617,1.553310,1.831228e+05,66846.300000,1.085627,1.521765
max,692674.000000,3.141130,3.186360e+06,1.276710e+06,4.141410,3.138540,3.587700e+06,1.146330e+06,4.559150,3.139200,2.800410e+06,788338.000000,4.798090,3.139020,2.503590e+06,481884.000000,4.730480,3.139660


### Let's first reproduce our benchmark linear model (with scaling) from Studio 4 so we have it for comparison; model = LinearSVC().

In [12]:
from sklearn.pipeline import make_pipeline #This allows one to build different steps together
piped_model = make_pipeline(StandardScaler(), LinearSVC(dual=False)) #make a pipeline with standard scaler and linear SVM
cv = StratifiedKFold(n_splits = 5, shuffle=True, random_state=101)# make a 5-fold stratified cross-validation, setting shuffle to "True" and random state to 101

benchmark_lim_piped = cross_validate(piped_model, features_lim, target, cv = cv, scoring = 'accuracy', return_train_score=True)

In [13]:
benchmark_lim_piped

{'fit_time': array([0.01308584, 0.00664091, 0.00556493, 0.00522804, 0.00447893]),
 'score_time': array([0.00189209, 0.0009551 , 0.00073695, 0.00072718, 0.00069976]),
 'test_score': array([0.894, 0.889, 0.89 , 0.892, 0.899]),
 'train_score': array([0.89575, 0.89525, 0.895  , 0.89825, 0.89275])}

In [14]:
np.round(benchmark_lim_piped['test_score'].mean(),3), np.round(benchmark_lim_piped['test_score'].std(), 3)

(np.float64(0.893), np.float64(0.004))

In [15]:
np.round(benchmark_lim_piped['train_score'].mean(),3), np.round(benchmark_lim_piped['train_score'].std(), 3)

(np.float64(0.895), np.float64(0.002))

### Parameter optimization 

When we optimize parameters with a grid search, we choose the parameters that give the best test scores. This is different from what would happen with new data - to do this fairly, at no point of the training procedure we are allowed to look at the test labels. Therefore, we would need to do <b> nested cross validation </b> to avoid leakage between the parameter optimization and the cross validation procedure and properly evaluate the generalization error.

If you don't do nested cross-validation, you'll unintentionally bias your generalization error estimate, as you'll be choosing the model you use based on the same test data you're using to evaluate the performance on new data. 

Since we're doing a lot of new things here, we'll first run an optimization <b> without </b> nested cross-validation. This is the approach you'd use to pick a set of optimal hyperparameters, which you could use to make predictions on new data.

Then, we'll run the same optimization <b> with </b> nested cross-validation, which is the approach used to get the correct generalization error. Then we'll compare the results from the two approaches. 

### Setup for both steps:

First we'll set up the model and hyperparameters to scan over, which are common to both approaches. We'll also set up 1 cross-validation, which will eventually be the outer one.

In [16]:
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=101) #1 layer of cross-validation

piped_model = make_pipeline(StandardScaler(), SVC()) #now using the general SVC so I can change the kernel

piped_model.get_params() #this shows how we can access parameters both for the scaler and the classifier


{'memory': None,
 'steps': [('standardscaler', StandardScaler()), ('svc', SVC())],
 'transform_input': None,
 'verbose': False,
 'standardscaler': StandardScaler(),
 'svc': SVC(),
 'standardscaler__copy': True,
 'standardscaler__with_mean': True,
 'standardscaler__with_std': True,
 'svc__C': 1.0,
 'svc__break_ties': False,
 'svc__cache_size': 200,
 'svc__class_weight': None,
 'svc__coef0': 0.0,
 'svc__decision_function_shape': 'ovr',
 'svc__degree': 3,
 'svc__gamma': 'scale',
 'svc__kernel': 'rbf',
 'svc__max_iter': -1,
 'svc__probability': False,
 'svc__random_state': None,
 'svc__shrinking': True,
 'svc__tol': 0.001,
 'svc__verbose': False}

Then we define a dictionary of parameter values that we'll run the optimization over.

You'll notice that we're not using a linear kernel. That's because the RBF (Gaussian) kernel with a very large $\gamma$ is equivalent to a linear kernel.

In [17]:
parameters = {'svc__kernel':['poly', 'rbf'], \
              'svc__gamma':[0.00001,'scale', 0.01, 0.1], 'svc__C':[0.1, 1.0, 10.0, 100.0], \
              'svc__degree': [2, 4, 8]}

### A few questions:
- Briefly describe what each hyperparameter in the dictionary does. 
- How many SVC's will be trained if we do 5-fold cross-validation for each combination of hyperparameters?
- How many of the SVC's trained correspond to models that are actually distinct? Hint: there are degeneracies! E.g. does the "degree" parameter change anything if you're using the rbf kernel?

>1. 'svc__kernel' chooses the type of function to run (either polynomial or rbf). 'svc__gamma' chooses how much curvature there is in the gaussian kernal fits. 'svc__C' chooses how much to penalize misclassifications. 'svc__degree' chooses the degree of the polynomial being fit. 
> 2. 5*4*3*4*2 = 140*8 = 1,120
> 3. 5*(4*3 + 4*4) = 140 unique 

### Now we'll run the optimization with just 1 layer of cross-validation.
Note that this might take a while (~1 min on my laptop); the early estimates output by this cell may be misleading because more complex models (in particular high gamma) take longer. To speed things up, we're running 4 jobs in parallel.

Notice that the $\texttt{GridSearchCV()}$ function constructs our $\texttt{model}$ object, but doesn't actually train any models! The training happens when we call $\texttt{fit}$, as usual.

Once you run this cell, the $\texttt{model}$ object will have attributes $\texttt{best\_score\_}$, $\texttt{best\_params\_}$ and $\texttt{best\_estimator\_}$, which give us access to the optimal estimator (printed out), as well as $\texttt{cv\_results\_}$ that can be used to visualize the performance of all models.

In [18]:
#optimizing SVC: THIS IS NOT YET NESTED CV

model = GridSearchCV(piped_model, parameters, cv = outer_cv, \
                     verbose = 2, n_jobs = 4, return_train_score=True)

In [19]:
model.fit(features_lim,target)

Fitting 5 folds for each of 96 candidates, totalling 480 fits
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=rbf; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=scale, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=scale, svc__kernel=poly; total time=   0.2s
[CV] END svc__C=0.1, svc__degre

,estimator,"Pipeline(step...svc', SVC())])"
,param_grid,"{'svc__C': [0.1, 1.0, ...], 'svc__degree': [2, 4, ...], 'svc__gamma': [1e-05, 'scale', ...], 'svc__kernel': ['poly', 'rbf']}"
,scoring,None
,n_jobs,4
,refit,True
,cv,StratifiedKFo... shuffle=True)
,verbose,2
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,True
,copy,True


In [20]:
print('Best params, best score:', "{:.4f}".format(model.best_score_), \
      model.best_params_)

Best params, best score: 0.8964 {'svc__C': 1.0, 'svc__degree': 2, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}


Which model performed the best? Give its relevant hyperparameters. What was its accuracy score?

> rbf did the best with svc__c of 1.0 and svc__gamma: scale. The accuracy score was 89.64%

#### We can visualize the models in a data frame, and rank them according to their test scores.

I like to look at the mean and std of the test scores, the mean of the train scores (so I can evaluate if they differ and the significance of the result), and also fitting time (we may pick a faster model instead of the best model if the scores are comparable)!

In [21]:
scores_lim = pd.DataFrame(model.cv_results_)

scores_lim.columns

Index(['mean_fit_time', 'std_fit_time', 'mean_score_time', 'std_score_time',
       'param_svc__C', 'param_svc__degree', 'param_svc__gamma',
       'param_svc__kernel', 'params', 'split0_test_score', 'split1_test_score',
       'split2_test_score', 'split3_test_score', 'split4_test_score',
       'mean_test_score', 'std_test_score', 'rank_test_score',
       'split0_train_score', 'split1_train_score', 'split2_train_score',
       'split3_train_score', 'split4_train_score', 'mean_train_score',
       'std_train_score'],
      dtype='object')

In [22]:
scores_lim[['params','mean_test_score','std_test_score','mean_train_score', \
            'mean_fit_time']].sort_values(by = 'mean_test_score', ascending = False)

,params,mean_test_score,std_test_score,mean_train_score,mean_fit_time
53,"{'svc__C': 10.0, 'svc__degree': 2, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}",0.8964,0.004587,0.91175,0.156922
35,"{'svc__C': 1.0, 'svc__degree': 4, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}",0.8964,0.005851,0.92135,0.161451
69,"{'svc__C': 10.0, 'svc__degree': 8, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}",0.8964,0.004587,0.91175,0.158280
27,"{'svc__C': 1.0, 'svc__degree': 2, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}",0.8964,0.005851,0.92135,0.163319
43,"{'svc__C': 1.0, 'svc__degree': 8, 'svc__gamma': 'scale', 'svc__kernel': 'rbf'}",0.8964,0.005851,0.92135,0.165954
61,"{'svc__C': 10.0, 'svc__degree': 4, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}",0.8964,0.004587,0.91175,0.161251
37,"{'svc__C': 1.0, 'svc__degree': 4, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}",0.8954,0.001855,0.90040,0.139743
29,"{'svc__C': 1.0, 'svc__degree': 2, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}",0.8954,0.001855,0.90040,0.141025
45,"{'svc__C': 1.0, 'svc__degree': 8, 'svc__gamma': 0.01, 'svc__kernel': 'rbf'}",0.8954,0.001855,0.90040,0.144935
39,"{'svc__C': 1.0, 'svc__degree': 4, 'svc__gamma': 0.1, 'svc__kernel': 'rbf'}",0.8938,0.008998,0.93975,0.206290


#### We can also isolate one type of kernel to look at it more closely.

In [23]:
scores_lim[scores_lim['param_svc__kernel'] == 'poly'][['params','mean_test_score','std_test_score',\
                        'mean_train_score','mean_fit_time']].sort_values(by = 'mean_test_score', ascending = False)

,params,mean_test_score,std_test_score,mean_train_score,mean_fit_time
50,"{'svc__C': 10.0, 'svc__degree': 2, 'svc__gamma': 'scale', 'svc__kernel': 'poly'}",0.8782,0.003487,0.88645,0.377970
54,"{'svc__C': 10.0, 'svc__degree': 2, 'svc__gamma': 0.1, 'svc__kernel': 'poly'}",0.8780,0.005099,0.88745,0.850029
76,"{'svc__C': 100.0, 'svc__degree': 2, 'svc__gamma': 0.01, 'svc__kernel': 'poly'}",0.8772,0.002561,0.88465,0.236923
30,"{'svc__C': 1.0, 'svc__degree': 2, 'svc__gamma': 0.1, 'svc__kernel': 'poly'}",0.8772,0.002561,0.88465,0.229838
74,"{'svc__C': 100.0, 'svc__degree': 2, 'svc__gamma': 'scale', 'svc__kernel': 'poly'}",0.8768,0.004578,0.88775,2.411590
26,"{'svc__C': 1.0, 'svc__degree': 2, 'svc__gamma': 'scale', 'svc__kernel': 'poly'}",0.8758,0.002786,0.88205,0.189542
78,"{'svc__C': 100.0, 'svc__degree': 2, 'svc__gamma': 0.1, 'svc__kernel': 'poly'}",0.8758,0.005492,0.88820,8.269450
6,"{'svc__C': 0.1, 'svc__degree': 2, 'svc__gamma': 0.1, 'svc__kernel': 'poly'}",0.8744,0.002245,0.87855,0.170361
52,"{'svc__C': 10.0, 'svc__degree': 2, 'svc__gamma': 0.01, 'svc__kernel': 'poly'}",0.8744,0.002245,0.87855,0.165017
58,"{'svc__C': 10.0, 'svc__degree': 4, 'svc__gamma': 'scale', 'svc__kernel': 'poly'}",0.8700,0.004690,0.96090,0.273920


### A few questions:
- Why do some of the models have identical scores (e.g., the top 3 models)?
- What hyperparameter values are common to all the best-performing models? 
- What hyperparameters do not strongly affect the accuracy? Ignore degenerate models in this discussion. 

>1. They only differ in a category that doesn't affect the outcome. 
>2. They are all polynomials of degree 2. 
>3. C does not strongly affect the score. 

### Next, we'll run nested cross-validation to get the generalization error.

First, we need to make one more layer of cross-validation. We'll do 4 splits for the inner layer.

In [24]:
inner_cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=10)


This time, the cross-validation applied to the hyperparameter optimization step is the $\texttt{inner\_cv}$.

Now, we'll use $\texttt{cross\_val\_score}$ with $\texttt{outer\_cv}$ (instead of $\texttt{fit}$!) to fit the model and get its score. 

This may be a bit confusing: remember that the $\texttt{GridSearchCV()}$ is just a constructor, it doesn't run any model fits! 


I found this explanation, from Arpit Omprakash on StackOverflow (https://stackoverflow.com/a/78544053) extremeley helpful:

"In the first line of code here, we are instantiating the GridSearchCV object using the inner_cv cross validator (but not fitting it). 

In the second line, we are doing a lot of things. First, using cross_val_score and outer_cv we break the initial data into different splits, let's call it x_tr_0, x_ts_0, x_tr_1, x_ts_1, x_tr_2, x_ts_2, x_tr_3, x_ts_3, x_tr_4, x_ts_4, (since there are five splits, and in each split, we have training and testing data). The training data is passed on to the GridSearchCV method in each fold. So, the inner_cv cross validator works on the training data splits from the outer_cv cross validator. So, in the GridSearchCV method, we are basically breaking down x_tr_0 into 4 splits: x_tr_0_0, x_tr_0_1, x_tr_0_2, x_tr_0_3. In these "inner" splits we are doing the hyperparameter tuning.

Once the optimal hyperparameters are calculated, we use these in the "outer" split for calculating the model performance. In this case, the model evaluation is done on the outer_cv split test data (which is unseen by the hyperparameter tuning "inner" split). This ensures that the performance values we are getting are more generalizable and there is no data leakage."

Because I'm using $\texttt{cross\_val\_score}$, all I'm returning is the scores of each of the 5 outer folds. If you want to return the models as well, you can do so with the $\texttt{cross\_validate}$ function.


Because we're running 5 times as many fits as before, this will take a bit longer! It ran in 1.5 minutes on my laptop.

In [25]:
nested_model = GridSearchCV(piped_model, parameters, cv = inner_cv, verbose = 2, n_jobs = 4, return_train_score=True)
nested_score = cross_val_score(nested_model, X=features_lim, y=target, cv= outer_cv, n_jobs=4)

Fitting 4 folds for each of 96 candidates, totalling 384 fits
Fitting 4 folds for each of 96 candidates, totalling 384 fits
Fitting 4 folds for each of 96 candidates, totalling 384 fits
Fitting 4 folds for each of 96 candidates, totalling 384 fits
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.4s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.3s
[CV] END svc__C=0.1, svc__degree=2, svc__gamma=1e-05, svc__kernel=poly; total time=   0.4s
[CV] END svc__C=0.1, svc

In [34]:
nested_score #an array of all the test scores for the outer CV, using the optimal hyperparameters from the inner CV step

array([0.895, 0.897, 0.897, 0.888, 0.898])

In [35]:
print(np.round(nested_score.mean(), 3), np.round(nested_score.std(), 3))

0.895 0.004


What is the average accuracy and generalization error of the model? Is the accuracy you found for the optimal model (without using nested cross-validation) compatible with this result?

>The average accuracy is 0.895. It is compatible with 0.8964 which was the accuracy found for the optimal model (without using nested cross-validation). The generalization error is 0.004. 

Results for a model with optimized hyperparameters are reported as: optimal model accuracy $\pm$ generalization error

Report your results for the optimized SVC model:

>0.895 +/- 0.004

### Diagnosis 

- Compare the performance of the best-performing model found in the optimization to the benchmark scaled Linear SVC model. Does it perform measurably better?

- Our diagnosis of the scaled Linear SVC model was that it had high bias. Has this problem been corrected by making the model more complex using different hyperparameters?



>0.898 > 0.8964 but not by much. The added hyperparameters did not significantly impact the high bias. 

The problem here is high bias, which is not that surprising given that we are using only a subset of features.

We can try two things: making up new features which might help, based on what we know about the problem, and using an imputing strategy to include information about the discarded features. Here we'll focus on adding new engineered features, and you'll try the imputing approach on Homework 2.

### Next step: Feature Engineering

First, we'll define some new variables.
We'll go back to the full list of features (not the abbreviated list we tested above) to develop our new engineered features.

In [36]:
features = features.fillna(0) #takes care of nan
features = features.replace('', 0) #takes care of empty string values
features.head()

,MET,METphi,P0_type,P0_E,P0_pt,P0_eta,P0_phi,P1_type,P1_E,P1_pt,P1_eta,P1_phi,P2_type,P2_E,P2_pt,P2_eta,P2_phi,P3_type,P3_E,P3_pt,P3_eta,P3_phi,P4_type,P4_E,P4_pt,P4_eta,P4_phi,P5_type,P5_E,P5_pt,P5_eta,P5_phi,P6_type,P6_E,P6_pt,P6_eta,P6_phi,P7_type,P7_E,P7_pt,P7_eta,P7_phi,P8_type,P8_E,P8_pt,P8_eta,P8_phi,P9_type,P9_E,P9_pt,P9_eta,P9_phi,P10_type,P10_E,P10_pt,P10_eta,P10_phi,P11_type,P11_E,P11_pt,P11_eta,P11_phi,P12_type,P12_E,P12_pt,P12_eta,P12_phi,P13_type,P13_E,P13_pt,P13_eta,P13_phi,P14_type,P14_E,P14_pt,P14_eta,P14_phi,P15_type,P15_E,P15_pt,P15_eta,P15_phi,P16_type,P16_E,P16_pt,P16_eta,P16_phi,P17_type,P17_E,P17_pt,P17_eta,P17_phi
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,62803.5,-1.810010,j,137571.0,128444.0,-0.345744,-0.307112,j,174209.0,127932.0,0.826569,2.332000,b,86788.9,84554.9,-0.180795,2.187970,j,140289.0,76955.8,-1.19933,-1.302800,m+,85230.6,70102.4,-0.645689,-1.659540,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,57594.2,-0.509253,j,161529.0,80458.3,-1.318010,1.402050,j,291490.0,68462.9,-2.126740,-2.582310,e-,44270.1,35139.6,-0.706120,-0.371392,e+,72883.9,26902.2,-1.65386,-3.129630,0,0.0,0.0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,82313.3,1.686840,b,167130.0,113078.0,0.937258,-2.068680,j,102423.0,54922.3,1.226850,0.646589,j,60768.9,36244.3,1.102890,-1.434480,j,77714.0,27801.5,1.68461,1.389690,j,26840.0,24469.3,-0.388937,-1.647260,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,30610.8,2.617120,j,112267.0,61383.9,-1.211050,-1.457800,b,40647.8,39472.0,-0.024646,-2.222800,j,201589.0,32978.6,-2.496040,1.137810,j,90096.7,26964.5,1.87132,0.817631,j,28235.4,25887.9,-0.411528,2.024290,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,45153.1,-2.241350,j,178174.0,100164.0,1.166880,-0.018721,j,92351.3,69762.1,0.774114,2.568740,j,61625.2,50086.7,0.652572,-3.012800,j,104193.0,31151.0,1.87641,0.865381,j,746585.0,26219.3,4.041820,-0.874169,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Let's start by looking at what kind of particles we have as a product of the collision.

In [37]:
# make a 2D numpy array of all the values of the particle type columns, storing the values as strings
ptypes = np.array([features['P'+str(i)+'_type'].values for i in range(0,18)])

print(ptypes)

print(np.shape(ptypes)) #note that the shape might be the transpose of what you expect! There are 18 rows (one for each particle), 5000 columns (one for each instance)

print(ptypes[0, 0:5]) #e.g. the type of particle 0 in instances 0 - 5

[['j' 'j' 'b' ... 'b' 'b' 'b']
 ['j' 'j' 'j' ... 'j' 'b' 'j']
 ['b' 'e-' 'j' ... 'j' 'j' 'j']
 ...
 [0.0 0.0 0.0 ... 0.0 0.0 0.0]
 [0.0 0.0 0.0 ... 0.0 0.0 0.0]
 [0.0 0.0 0.0 ... 0.0 0.0 0.0]]
(18, 5000)
['j' 'j' 'b' 'j' 'j']


In [38]:
#list the unique values of the particle type columns
np.unique(ptypes.astype('str'))

array(['0', '0.0', 'b', 'e+', 'e-', 'g', 'j', 'm+', 'm-'], dtype='<U3')

#### Here are the proposed new features (justification can be found in Chapter 4).
    
    1. The total number of particles produced
    2. The total number of b jets
    3. The total number of jets
    4. The total number of leptons (electrons, positron, mu+, mu-)

In [39]:
#count number of non-zero types for each instance

ntot = np.array([(np.sum(np.array([ptypes[i][j] != 0 for i in range(ptypes.shape[0])]))) for j in range(features.shape[0])])
ntot

array([5, 4, 5, ..., 5, 9, 5], shape=(5000,))

In [40]:
#define new column in my data frame
features['Total_products'] = ntot

In [41]:
#count number of b jets 
nbtot = np.array([np.sum(np.array([ptypes[i][j] == 'b' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])
#define new column in my data frame
features['Total_b'] = nbtot

In [42]:
#You get the idea, let's count all types (jets, photons g, e-, e+, mu-, mu+)
njtot = np.array([np.sum(np.array([ptypes[i][j] == 'j' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])
ngtot = np.array([np.sum(np.array([ptypes[i][j] == 'g' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])

# count each of the lepton types separately, then sum to get total leptons
n_el_tot = np.array([np.sum(np.array([ptypes[i][j] == 'e-' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])
n_pos_tot = np.array([np.sum(np.array([ptypes[i][j] == 'e+' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])
n_muneg_tot = np.array([np.sum(np.array([ptypes[i][j] == 'm-' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])
n_mupos_tot = np.array([np.sum(np.array([ptypes[i][j] == 'm+' for i in range(ptypes.shape[0])])) for j in range(features.shape[0])])
n_lepton_tot = n_el_tot + n_pos_tot + n_muneg_tot + n_mupos_tot

And here we define the other new features:

In [43]:
features['Total_j'] = njtot
features['Total_g'] = ngtot
features['Total_leptons'] = n_lepton_tot

In [44]:
features.head() #scroll to the final columns to see your new features

,MET,METphi,P0_type,P0_E,P0_pt,P0_eta,P0_phi,P1_type,P1_E,P1_pt,P1_eta,P1_phi,P2_type,P2_E,P2_pt,P2_eta,P2_phi,P3_type,P3_E,P3_pt,P3_eta,P3_phi,P4_type,P4_E,P4_pt,P4_eta,P4_phi,P5_type,P5_E,P5_pt,P5_eta,P5_phi,P6_type,P6_E,P6_pt,P6_eta,P6_phi,P7_type,P7_E,P7_pt,P7_eta,P7_phi,P8_type,P8_E,P8_pt,P8_eta,P8_phi,P9_type,P9_E,P9_pt,P9_eta,P9_phi,P10_type,P10_E,P10_pt,P10_eta,P10_phi,P11_type,P11_E,P11_pt,P11_eta,P11_phi,P12_type,P12_E,P12_pt,P12_eta,P12_phi,P13_type,P13_E,P13_pt,P13_eta,P13_phi,P14_type,P14_E,P14_pt,P14_eta,P14_phi,P15_type,P15_E,P15_pt,P15_eta,P15_phi,P16_type,P16_E,P16_pt,P16_eta,P16_phi,P17_type,P17_E,P17_pt,P17_eta,P17_phi,Total_products,Total_b,Total_j,Total_g,Total_leptons
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,62803.5,-1.810010,j,137571.0,128444.0,-0.345744,-0.307112,j,174209.0,127932.0,0.826569,2.332000,b,86788.9,84554.9,-0.180795,2.187970,j,140289.0,76955.8,-1.19933,-1.302800,m+,85230.6,70102.4,-0.645689,-1.659540,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,3,0,1
1,57594.2,-0.509253,j,161529.0,80458.3,-1.318010,1.402050,j,291490.0,68462.9,-2.126740,-2.582310,e-,44270.1,35139.6,-0.706120,-0.371392,e+,72883.9,26902.2,-1.65386,-3.129630,0,0.0,0.0,0.000000,0.000000,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,2,0,2
2,82313.3,1.686840,b,167130.0,113078.0,0.937258,-2.068680,j,102423.0,54922.3,1.226850,0.646589,j,60768.9,36244.3,1.102890,-1.434480,j,77714.0,27801.5,1.68461,1.389690,j,26840.0,24469.3,-0.388937,-1.647260,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,4,0,0
3,30610.8,2.617120,j,112267.0,61383.9,-1.211050,-1.457800,b,40647.8,39472.0,-0.024646,-2.222800,j,201589.0,32978.6,-2.496040,1.137810,j,90096.7,26964.5,1.87132,0.817631,j,28235.4,25887.9,-0.411528,2.024290,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,4,0,0
4,45153.1,-2.241350,j,178174.0,100164.0,1.166880,-0.018721,j,92351.3,69762.1,0.774114,2.568740,j,61625.2,50086.7,0.652572,-3.012800,j,104193.0,31151.0,1.87641,0.865381,j,746585.0,26219.3,4.041820,-0.874169,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,0,5,0,0


### Feature engineering 1: impact of engineered variables

We'll add these 4 new features to our original set to see if that improves our classifier.

In [45]:
features_lim_2 = features[['MET', 'METphi', 'P0_E', 'P0_pt', 'P0_eta', 'P0_phi', 
                           'P1_E', 'P1_pt', 'P1_eta', 'P1_phi', 
                           'P2_E', 'P2_pt', 'P2_eta', 'P2_phi', 
                           'P3_E', 'P3_pt', 'P3_eta', 'P3_phi',
                           'Total_products', 'Total_b' ,'Total_j','Total_g','Total_leptons']]

First, we'll try our benchmark model (Standard Scaler and Linear SVC), using 5-fold cross-validation

In [46]:
piped_model #remember our benchmark model?

,steps,"[('standardscaler', ...), ('svc', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,C,1.0
,kernel,'rbf'
,degree,3
,gamma,'scale'


In [47]:
benchmark_lim2 = cross_validate(piped_model, features_lim_2, target, cv = cv, scoring = 'accuracy', return_train_score=True)

In [48]:
benchmark_lim2

{'fit_time': array([0.16265488, 0.113482  , 0.11812305, 0.11666703, 0.11458993]),
 'score_time': array([0.04477811, 0.0421741 , 0.04326773, 0.04346395, 0.04190898]),
 'test_score': array([0.942, 0.933, 0.952, 0.945, 0.938]),
 'train_score': array([0.96425, 0.96725, 0.966  , 0.96475, 0.96725])}

In [49]:
np.round(benchmark_lim2['test_score'].mean(),3), np.round(benchmark_lim2['test_score'].std(), 3)

(np.float64(0.942), np.float64(0.006))

In [50]:
np.round(benchmark_lim2['train_score'].mean(),3), np.round(benchmark_lim2['train_score'].std(), 3)

(np.float64(0.966), np.float64(0.001))

What is the average accuracy and generalization error of the model? Note: we haven't done any parameter optimization here, so we don't need nested cross-validation to get the error!

>Average accuracy: 0.942. Generalization error: 0.006

Compare the performance of this enhanced-feature Linear SVC to your optimized model. Which change had the larger impact on the bias: hyperparameter optimization or feature engineering?

>Feature engineering had a larger impact on bias because 0.942 is significantly better than 0.895

### This is a very significant improvement, which cuts our error rate in half!

In my experience, this knowledge-informed feature engineering is often very successful, more than hyperparameter optimization. Machine learning methods are often tooted for their ability to learn relevant representations, but non-deep-learning methods are less capable to do so, and providing informative features is very helpful.

We can optimize this model as well, just as we did before. For the moment we'll skip this for the sake of time, but you'll get lots of practice with this on HW 2.

### Techniques for Feature Engineering: One-Hot Encoding

Another feature engineering attempt we could potentially do is use the type of product in the i-th location as a feature. To do this, we need to somehow turn the particle types into numerical features, since that's all SVM's know how to handle.

We could do it with label encoding, as we did earlier in this notebook, but such strategy introduces a notion of distance metric (labels that are mapped to 0 and 1 are interpreted to be closer to each other than labels that are mapped into 0 and 7). 

As an alternative, we can introduce as many new columns as possible values for each categorical variable we are re-mapping, and we just use a 0/1 to indicate that the particle is of that type. This is known as "one-hot encoding," since only one of the categorical variable columns we add can ever be "hot" at a time (that is to say, a 1, instead of a 0). 

This is achieved with the wonderfully-named "get_dummies" function:

In [51]:
features_add = pd.get_dummies(data=features, columns=['P'+str(i)+'_type' for i in range(0,18)])

In [52]:
features_add.columns[77:90] #A subset of the new features

Index(['Total_g', 'Total_leptons', 'P0_type_b', 'P0_type_j', 'P1_type_0',
       'P1_type_b', 'P1_type_e+', 'P1_type_e-', 'P1_type_g', 'P1_type_j',
       'P1_type_m+', 'P1_type_m-', 'P2_type_0'],
      dtype='object')

Notice a couple of things:
- Particle 0 only comes in type "b" or "j" -- this probably has to do with how the ATLAS collaboration decides to trigger on events. Presumably they're only storing events that have at least 1 jet in them (whether or not it's a b-jet). 
-  Empty particle tracks are now listed as "type 0" in the encoding scheme. This may be acceptable, or it could be something you decide to clean up on a later test, if you notice problems classifying events with missing tracks. Developing an ML method is often an iterative process! You don't need to have everything perfectly correct the first time through; try it and see what happens before investing a lot of time into perfecting things. 

In [53]:
features_add.shape

features_add

,MET,METphi,P0_E,P0_pt,P0_eta,P0_phi,P1_E,P1_pt,P1_eta,P1_phi,P2_E,P2_pt,P2_eta,P2_phi,P3_E,P3_pt,P3_eta,P3_phi,P4_E,P4_pt,P4_eta,P4_phi,P5_E,P5_pt,P5_eta,P5_phi,P6_E,P6_pt,P6_eta,P6_phi,P7_E,P7_pt,P7_eta,P7_phi,P8_E,P8_pt,P8_eta,P8_phi,P9_E,P9_pt,P9_eta,P9_phi,P10_E,P10_pt,P10_eta,P10_phi,P11_E,P11_pt,P11_eta,P11_phi,P12_E,P12_pt,P12_eta,P12_phi,P13_E,P13_pt,P13_eta,P13_phi,P14_E,P14_pt,P14_eta,P14_phi,P15_E,P15_pt,P15_eta,P15_phi,P16_E,P16_pt,P16_eta,P16_phi,P17_E,P17_pt,P17_eta,P17_phi,Total_products,Total_b,Total_j,Total_g,Total_leptons,P0_type_b,P0_type_j,P1_type_0,P1_type_b,P1_type_e+,P1_type_e-,P1_type_g,P1_type_j,P1_type_m+,P1_type_m-,P2_type_0,P2_type_b,P2_type_e+,P2_type_e-,P2_type_g,P2_type_j,P2_type_m+,P2_type_m-,P3_type_0,P3_type_b,P3_type_e+,P3_type_e-,P3_type_g,P3_type_j,P3_type_m+,P3_type_m-,P4_type_0,P4_type_b,P4_type_e+,P4_type_e-,P4_type_g,P4_type_j,P4_type_m+,P4_type_m-,P5_type_0,P5_type_b,P5_type_e+,P5_type_e-,P5_type_g,P5_type_j,P5_type_m+,P5_type_m-,P6_type_0,P6_type_b,P6_type_e+,P6_type_e-,P6_type_g,P6_type_j,P6_type_m+,P6_type_m-,P7_type_0,P7_type_b,P7_type_e+,P7_type_e-,P7_type_g,P7_type_j,P7_type_m+,P7_type_m-,P8_type_0,P8_type_b,P8_type_e+,P8_type_e-,P8_type_g,P8_type_j,P8_type_m+,P8_type_m-,P9_type_0,P9_type_b,P9_type_e+,P9_type_e-,P9_type_g,P9_type_j,P9_type_m+,P9_type_m-,P10_type_0,P10_type_b,P10_type_e+,P10_type_e-,P10_type_g,P10_type_j,P10_type_m+,P10_type_m-,P11_type_0,P11_type_b,P11_type_e+,P11_type_e-,P11_type_j,P11_type_m+,P11_type_m-,P12_type_0,P12_type_b,P12_type_e+,P12_type_e-,P12_type_g,P12_type_j,P12_type_m+,P12_type_m-,P13_type_0,P13_type_b,P13_type_j,P14_type_0,P14_type_b,P14_type_j,P15_type_0.0,P16_type_0.0,P17_type_0.0
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,62803.5,-1.810010,137571.0,128444.0,-0.345744,-0.307112,174209.0,127932.0,0.826569,2.332000,86788.9,84554.9,-0.180795,2.187970,140289.0,76955.8,-1.199330,-1.302800,85230.6,70102.4,-0.645689,-1.659540,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5,1,3,0,1,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,True,True,True
1,57594.2,-0.509253,161529.0,80458.3,-1.318010,1.402050,291490.0,68462.9,-2.126740,-2.582310,44270.1,35139.6,-0.706120,-0.371392,72883.9,26902.2,-1.653860,-3.129630,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0,0.00000,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4,0,2,0,2,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,Fals

### Feature engineering 2: add other variables (type of product) for the first four particles.

In [54]:
features_lim_3 = features_add[['MET', 'METphi', 'P0_E', 'P0_pt', 'P0_eta', 'P0_phi', 
                           'P1_E', 'P1_pt', 'P1_eta', 'P1_phi', 
                           'P2_E', 'P2_pt', 'P2_eta', 'P2_phi', 
                           'P3_E', 'P3_pt', 'P3_eta', 'P3_phi',
                           'Total_products', 'Total_b' ,'Total_j','Total_g','Total_leptons',
                           'P0_type_b', 'P0_type_j',
                            'P1_type_0', 'P1_type_b', 'P1_type_e+', 'P1_type_e-', 'P1_type_g', 'P1_type_j', 'P1_type_m+', 'P1_type_m-', 
                            'P2_type_0', 'P2_type_b', 'P2_type_e+', 'P2_type_e-', 'P2_type_g', 'P2_type_j', 'P2_type_m+', 'P2_type_m-', 
                            'P3_type_0', 'P3_type_b', 'P3_type_e+', 'P3_type_e-', 'P3_type_g', 'P3_type_j', 'P3_type_m+', 'P3_type_m-']]

In [55]:
features_lim_3.head()

,MET,METphi,P0_E,P0_pt,P0_eta,P0_phi,P1_E,P1_pt,P1_eta,P1_phi,P2_E,P2_pt,P2_eta,P2_phi,P3_E,P3_pt,P3_eta,P3_phi,Total_products,Total_b,Total_j,Total_g,Total_leptons,P0_type_b,P0_type_j,P1_type_0,P1_type_b,P1_type_e+,P1_type_e-,P1_type_g,P1_type_j,P1_type_m+,P1_type_m-,P2_type_0,P2_type_b,P2_type_e+,P2_type_e-,P2_type_g,P2_type_j,P2_type_m+,P2_type_m-,P3_type_0,P3_type_b,P3_type_e+,P3_type_e-,P3_type_g,P3_type_j,P3_type_m+,P3_type_m-
ID,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
0,62803.5,-1.810010,137571.0,128444.0,-0.345744,-0.307112,174209.0,127932.0,0.826569,2.332000,86788.9,84554.9,-0.180795,2.187970,140289.0,76955.8,-1.19933,-1.302800,5,1,3,0,1,False,True,False,False,False,False,False,True,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False
1,57594.2,-0.509253,161529.0,80458.3,-1.318010,1.402050,291490.0,68462.9,-2.126740,-2.582310,44270.1,35139.6,-0.706120,-0.371392,72883.9,26902.2,-1.65386,-3.129630,4,0,2,0,2,False,True,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False
2,82313.3,1.686840,167130.0,113078.0,0.937258,-2.068680,102423.0,54922.3,1.226850,0.646589,60768.9,36244.3,1.102890,-1.434480,77714.0,27801.5,1.68461,1.389690,5,1,4,0,0,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False
3,30610.8,2.617120,112267.0,61383.9,-1.211050,-1.457800,40647.8,39472.0,-0.024646,-2.222800,201589.0,32978.6,-2.496040,1.137810,90096.7,26964.5,1.87132,0.817631,5,1,4,0,0,False,True,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False
4,45153.1,-2.241350,178174.0,100164.0,1.166880,-0.018721,92351.3,69762.1,0.774114,2.568740,61625.2,50086.7,0.652572,-3.012800,104193.0,31151.0,1.87641,0.865381,5,0,5,0,0,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False


Let's train the benchmark model one more time, using the new features.

In [56]:
benchmark_lim3 = cross_validate(piped_model, features_lim_3, target, cv = cv, scoring = 'accuracy', return_train_score=True)

In [57]:
benchmark_lim3

{'fit_time': array([0.15922618, 0.13249683, 0.14029288, 0.13724089, 0.13480401]),
 'score_time': array([0.05939198, 0.05910897, 0.05959821, 0.06332421, 0.06202197]),
 'test_score': array([0.943, 0.935, 0.959, 0.944, 0.937]),
 'train_score': array([0.9595 , 0.9635 , 0.96025, 0.96025, 0.96125])}

In [58]:
np.round(benchmark_lim3['test_score'].mean(),3), np.round(benchmark_lim3['test_score'].std(), 3)

(np.float64(0.944), np.float64(0.008))

In [59]:
np.round(benchmark_lim3['train_score'].mean(),3), np.round(benchmark_lim3['train_score'].std(), 3)

(np.float64(0.961), np.float64(0.001))

What is the average accuracy and generalization error of the model? Note: we haven't done any parameter optimization here, so we don't need nested cross-validation to get the error!

>Average accuracy: 0.944. Generalization error: 0.008. 

Compare the performance of this enhanced-feature Linear SVC to the first feature-engineered model (the one with the numbers of products as added features). Do you see any improvement?

>The small improvement is within the standard deviation so it is negligable. 

#### Next, we would normally optimize the model. Again, we'll skip this for the sake of time in this Studio.

### Finally, we can try with all the features.

In [60]:
features_add.shape

(5000, 185)

In [61]:
benchmark_all = cross_validate(piped_model, features_add, target, cv = cv, scoring = 'accuracy', return_train_score=True)

In [62]:
benchmark_all

{'fit_time': array([0.38969016, 0.33312297, 0.33681417, 0.33290219, 0.32768393]),
 'score_time': array([0.10464787, 0.10317111, 0.10529685, 0.10355783, 0.10307527]),
 'test_score': array([0.941, 0.936, 0.952, 0.942, 0.932]),
 'train_score': array([0.9695 , 0.969  , 0.967  , 0.97   , 0.96775])}

In [63]:
np.round(benchmark_all['test_score'].mean(),3), np.round(benchmark_all['test_score'].std(), 3)

(np.float64(0.941), np.float64(0.007))

In [64]:
np.round(benchmark_all['train_score'].mean(),3), np.round(benchmark_all['train_score'].std(), 3)

(np.float64(0.969), np.float64(0.001))

What is the average accuracy and generalization error of the model? Note: we haven't done any parameter optimization here, so we don't need nested cross-validation to get the error!'

>Average accuracy: 0.941. Generalization error: 0.007. 

Compare the bias and variance of this model (using all the features) to the other two models using engineered features. What happened to the variance as the number of features increased?

>The variance went up as the number of features increased (0.028 for all compared to 0.024 for lim 2 and 0.017 for lim 3). The accuracy went slightly down but by a margin within the standard deviation. 

We could run the optimization, but as you might have anticipated, it won't help much, and it is very time consuming.

### Take-home message: feature engineering often works best if we use subject matter knowledge, and buulding more features is not necessarily better.

### Acknowledgement Statement:

### You're done! Upload your work to Gradescope.